# Week 3 — RAG Pipeline

This notebook demonstrates the full end-to-end RAG pipeline built this week.

**Pipeline:**
```
New ticket description
    → ChromaDB vector search (top 20)
    → Cross-encoder reranker (top 5)
    → Mistral via Ollama
    → Root cause + suggested resolution + confidence
```

**Source files built this week:**
- `src/retriever.py` — loads ChromaDB and runs similarity search
- `src/reranker.py` — cross-encoder reranking with `ms-marco-MiniLM-L-6-v2`
- `src/rag_chain.py` — LangChain chain wiring retriever → reranker → LLM

> ChromaDB was populated in Week 2 (`src/embeddings.py`). We don't re-embed here.

## 1. Setup

In [8]:
import sys
import os

# Add the project root to sys.path so `src.*` imports resolve correctly
# when running from the notebooks/ directory.
sys.path.insert(0, os.path.abspath(".."))
os.chdir("..")  # set working directory to project root so data/ paths resolve

from src.retriever import retrieve
from src.reranker import rerank
from src.rag_chain import run

print("Imports OK")

Imports OK


## 2. Retriever

`retrieve()` embeds the query with BGE and searches ChromaDB.
We fetch `top_k=20` here — more than we need — because the reranker will trim this down.
The `distance` field is the L2 distance between vectors: lower = more similar.

In [9]:
query = "My laptop won't connect to the VPN and I keep getting an authentication error"

candidates = retrieve(query, top_k=20)

print(f"Retrieved {len(candidates)} candidates\n")
for i, t in enumerate(candidates[:5], 1):  # print top 5 for inspection
    print(f"{i}. [{t['category']}] {t['subject']}")
    print(f"   Distance: {t['distance']:.4f}")
    print(f"   Resolution: {t['resolution'][:120]}...\n")

InternalError: Read-only file system (os error 30)

## 3. Reranker

`rerank()` scores every `(query, resolution)` pair using the cross-encoder.
Unlike vector distance, the cross-encoder reads both texts together —
it can catch relevance that pure embedding similarity misses.

Notice the order may change from the retriever results above:
a ticket ranked 8th by vector search might jump to 1st after reranking.

In [ ]:
top_tickets = rerank(query, candidates, top_k=5)

print(f"Top {len(top_tickets)} after reranking:\n")
for i, t in enumerate(top_tickets, 1):
    print(f"{i}. [{t['category']}] {t['subject']}")
    print(f"   Rerank score: {t['rerank_score']:.4f}  |  Original distance: {t['distance']:.4f}")
    print(f"   Resolution: {t['resolution'][:120]}...\n")

## 4. Full RAG chain — single query

`run()` executes all steps in sequence and returns:
- `response` — the LLM's generated resolution suggestion
- `sources` — the 5 tickets that were passed as context (for transparency)

Model: **Mistral 7B via Ollama** (running locally on MPS).
First call will be slower as the model loads into memory.

In [ ]:
result = run(query, provider="ollama", model_name="mistral")

print("=" * 60)
print("QUERY")
print("=" * 60)
print(result["query"])

print("\n" + "=" * 60)
print("RESPONSE")
print("=" * 60)
print(result["response"])

print("\n" + "=" * 60)
print("SOURCES USED AS CONTEXT")
print("=" * 60)
for i, src in enumerate(result["sources"], 1):
    print(f"{i}. {src['subject']} (score: {src['rerank_score']:.3f})")

## 5. Multiple examples

Running 4 queries across different ticket categories to verify the pipeline
generalises — not just good at one type of ticket.

In [ ]:
example_queries = [
    "Outlook keeps crashing when I try to open attachments larger than 5MB",
    "New employee needs laptop provisioned and access to the HR system",
    "The shared printer on floor 3 is offline and nobody can print",
    "I accidentally deleted a folder from the shared drive, can it be recovered?",
]

for query in example_queries:
    result = run(query, provider="ollama", model_name="mistral")
    print("\n" + "=" * 60)
    print(f"QUERY: {result['query']}")
    print("=" * 60)
    print(result["response"])
    print(f"\nSources: {', '.join(s['subject'][:40] for s in result['sources'])}")

## Summary

| Stage | Tool | Purpose |
|---|---|---|
| Embedding | `BAAI/bge-small-en` | Encode tickets + queries into vectors |
| Vector search | ChromaDB | Fast approximate nearest-neighbour retrieval |
| Reranking | `ms-marco-MiniLM-L-6-v2` | Precise relevance scoring on top-20 candidates |
| Generation | Mistral 7B (Ollama) | Synthesise a resolution from retrieved context |

**Week 4:** Fine-tune a classifier for ticket category + severity using scikit-learn → DistilBERT, tracked with MLflow.